# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnzilaAhsan/week1-asm1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


1. **One row = one content item, on one day.** Table: `fact_content_daily_performance`,
   grain `report_date × client_hash_id × content_hash_id`.
2. **Table(s) used:** `fact_content_daily_performance` only, for this notebook — filtered to the
   `month=2026-03` partition (a mid-panel month, not the sealed final month).
3. **Time window:** March 2026 only (`2026-03-01` to `2026-03-31`), split in half within the
   notebook: days 1–15 as the "known" half, days 16–31 as the "outcome" half — used to build a
   self-contained proxy without touching June (the `_sample` table / final month), which the
   assignment brief flags as a sealed test month, not a source of label logic.

In [2]:
# Setup — connect DuckDB to the warehouse. Requires a Hugging Face READ token
# (plain Read type, not fine-grained) with access granted on FlyRank/internship-warehouse.
# Store it as a Colab Secret named HF_TOKEN — never paste a token into a cell.

%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
print("Pointed at month=2026-03 (mid-panel) — not the _sample table, per the assignment warning.")


Paste your Hugging Face READ token (hf_...): ··········
Pointed at month=2026-03 (mid-panel) — not the _sample table, per the assignment warning.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. What I'd predict, and one thing I exclude

4. **Label / proxy:** whether a content item's impressions in the *outcome half* of March
   (days 16–31) drop by 20%+ versus its *known half* (days 1–15). This is my own within-month
   proxy for "at risk of decline," built from raw daily impressions — not from any precomputed
   trend column (none exists at this grain, and the flyrank data skill flags precomputed
   trend fields as label material, never features).
5. **One thing I deliberately exclude:** GA4 engagement columns for any row where
   `ga4_data_available` is not `TRUE`. That flag is three-valued (TRUE / FALSE / NULL) — treating
   a FALSE or NULL row as "zero engagement" would silently fabricate a signal that was never
   actually measured, so those rows are excluded from engagement features rather than filled.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 — grain.** Zero rows back confirms one row really is one (day, client, content).

In [4]:
# Query 1: grain check
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {MARCH}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Duplicate (date, client, content) rows: {len(dupes)}  (0 = grain confirmed)")
dupes


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (date, client, content) rows: 0  (0 = grain confirmed)


,report_date,client_hash_id,content_hash_id,c


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


**Query 2 — row count and date span.** Does March actually look like March?

In [5]:
# Query 2: row count + date span for this slice
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS first_date, MAX(report_date) AS last_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {MARCH}
""").df()
span


,n_rows,first_date,last_date,n_clients,n_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


**Query 3 — availability, filtered with `IS TRUE`.** How many rows actually have usable
GA4 data, versus how many exist in total?

In [6]:
# Query 3: availability check — IS TRUE (not = TRUE) because the flag is 3-valued
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_flagged_false_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NULL)  AS ga4_null_flag_rows
    FROM {MARCH}
""").df()
print(avail)
print(f"\n{avail['ga4_available_rows'][0]:,} of {avail['total_rows'][0]:,} rows "
      f"({avail['ga4_available_rows'][0] / avail['total_rows'][0] * 100:.1f}%) "
      f"survive an IS TRUE filter on ga4_data_available.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows  ga4_flagged_false_rows  ga4_null_flag_rows
0     9841378              413966                 6408671             3018741

413,966 of 9,841,378 rows (4.2%) survive an IS TRUE filter on ga4_data_available.


### Five features (max), each knowable at the decision moment

Decision moment = end of the known half (day 15). Every feature below is built **only** from
days 1–15, so nothing here could see into the outcome half used for the label.


In [7]:
# Build the 5-feature frame from the "known" half of March (days 1-15) only.
known = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions)                                   AS f1_impressions,
        SUM(gsc_clicks)                                        AS f2_clicks,
        AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS f3_avg_position,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS f4_days_with_impressions,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) * 100 AS f5_ctr_pct
    FROM {MARCH}
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING f1_impressions >= 20   -- floor: need real prior demand to trust a ratio off it
""").df()

print(f"{len(known):,} content items with enough known-half demand")
known.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

109,592 content items with enough known-half demand


,client_hash_id,content_hash_id,f1_impressions,f2_clicks,f3_avg_position,f4_days_with_impressions,f5_ctr_pct
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,111.0,0.0,5.222776,13,0.000000
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,38.0,1.0,4.638889,9,2.631579
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,219.0,1.0,3.737399,15,0.456621
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,20.0,0.0,3.597222,9,0.000000
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,1494.0,0.0,6.156643,14,0.000000


**Available-when line, one per feature:**
1. `f1_impressions` — sum of GSC impressions, days 1–15. Known the moment day 15 closes.
2. `f2_clicks` — sum of GSC clicks, days 1–15. Same window, same reason.
3. `f3_avg_position` — mean GSC position, days 1–15 only. A running average, updatable daily —
   nothing here depends on days 16–31.
4. `f4_days_with_impressions` — count of days 1–15 with ≥1 impression. Pure count over the known
   window.
5. `f5_ctr_pct` — clicks/impressions over days 1–15. A ratio of two known-half sums; still fully
   computable before day 16 starts.


### The trap: add one label-derived column on purpose

Label: did impressions drop ≥20% from the known half (days 1–15) to the outcome half
(days 16–31)? I build the label from the *outcome* half — then, on purpose, add the outcome
half's raw impressions as an extra "feature" and watch the score jump toward perfect.


In [8]:
# Build the label from the OUTCOME half (days 16-31) and join it to the known-half features.
outcome = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS outcome_impressions
    FROM {MARCH}
    WHERE report_date > DATE '2026-03-15'
    GROUP BY 1, 2
""").df()

data = known.merge(outcome, on=["client_hash_id", "content_hash_id"], how="left")
data["outcome_impressions"] = data["outcome_impressions"].fillna(0)
data["declined"] = (data["outcome_impressions"] < 0.8 * data["f1_impressions"]).astype(int)
print(f"declined rate: {data['declined'].mean()*100:.1f}%  ({data['declined'].sum():,} of {len(data):,})")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

declined rate: 29.1%  (31,901 of 109,592)


In [9]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

honest_features = ["f1_impressions", "f2_clicks", "f3_avg_position", "f4_days_with_impressions", "f5_ctr_pct"]
X = data[honest_features].fillna(0)
y = data["declined"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

honest_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_tr, y_tr)
honest_acc = accuracy_score(y_te, honest_tree.predict(X_te))
honest_auc = roc_auc_score(y_te, honest_tree.predict_proba(X_te)[:, 1])
print(f"HONEST (5 known-half features only)   accuracy={honest_acc:.3f}  auc={honest_auc:.3f}")

# --- Now the trap: add the label-derived column on purpose ---
X_leaky = data[honest_features + ["outcome_impressions"]].fillna(0)
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)
leaky_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(Xl_tr, yl_tr)
leaky_acc = accuracy_score(yl_te, leaky_tree.predict(Xl_te))
leaky_auc = roc_auc_score(yl_te, leaky_tree.predict_proba(Xl_te)[:, 1])
print(f"LEAKY (+ outcome_impressions, the label's own input)  accuracy={leaky_acc:.3f}  auc={leaky_auc:.3f}  <- jumps toward perfect")

print()
print("Leaky tree's top split:")
print(export_text(leaky_tree, feature_names=honest_features + ["outcome_impressions"]))


HONEST (5 known-half features only)   accuracy=0.496  auc=0.605
LEAKY (+ outcome_impressions, the label's own input)  accuracy=0.685  auc=0.762  <- jumps toward perfect

Leaky tree's top split:
|--- outcome_impressions <= 35.50
|   |--- outcome_impressions <= 20.50
|   |   |--- outcome_impressions <= 17.50
|   |   |   |--- class: 1
|   |   |--- outcome_impressions >  17.50
|   |   |   |--- class: 1
|   |--- outcome_impressions >  20.50
|   |   |--- f1_impressions <= 32.50
|   |   |   |--- class: 0
|   |   |--- f1_impressions >  32.50
|   |   |   |--- class: 1
|--- outcome_impressions >  35.50
|   |--- f4_days_with_impressions <= 13.50
|   |   |--- f1_impressions <= 47.50
|   |   |   |--- class: 0
|   |   |--- f1_impressions >  47.50
|   |   |   |--- class: 0
|   |--- f4_days_with_impressions >  13.50
|   |   |--- outcome_impressions <= 550.50
|   |   |   |--- class: 1
|   |   |--- outcome_impressions >  550.50
|   |   |   |--- class: 0



`outcome_impressions` is literally the number the label is computed from, so the tree just
splits on it and "solves" the label — that's not a discovered pattern, it's the answer smuggled
in as a feature. **Deleting it** and keeping only the 5 known-half features restores the honest,
lower score above — that's the number I'd actually report.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


**One named limitation:** this whole notebook runs on a single mid-panel month (March 2026) for
one client-slice of content items, split in half to simulate a known/outcome window. A real
prev-30/last-30 label built across month *boundaries* (e.g. late Feb vs early April) would see
different seasonality and a different client mix than this artificial within-March split — so the
declined-rate and score numbers above describe *this month's internal split*, not a claim about
March-to-April behavior or about any other month in the panel.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
print(dupes.to_string())
print(span.to_string())
print(known.head().to_string())

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []
    n_rows first_date  last_date  n_clients  n_content_items
0  9841378 2026-03-01 2026-03-31         55           331437
            client_hash_id           content_hash_id  f1_impressions  f2_clicks  f3_avg_position  f4_days_with_impressions  f5_ctr_pct
0  client_62f4a7e64f5e0096  content_d0dff76c889de68f           111.0        0.0         5.222776                        13    0.000000
1  client_62f4a7e64f5e0096  content_67741cce996cfafa            38.0        1.0         4.638889                         9    2.631579
2  client_62f4a7e64f5e0096  content_2e6360ad20fd7107           219.0        1.0         3.737399                        15    0.456621
3  client_62f4a7e64f5e0096  content_ac8663da7484669a            20.0        0.0         3.597222                         9    0.000000
4  client_62f4a7e64f5e0096  content_65c50dfe9d87a585          1494.0        0.0         6.156643                      

## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.